In [1]:
from app.models import Embedding, SubTopic, Bookmark, Document_Entity_Link, Entity, Document
from app.db_connector import get_engine
import app.getters as getter
import app.icog_util as util
from app.prompt_models import ResponseWithIndex, DocumentPromptVerbatim
from app.transformers_util import generate_embeddings, get_util
from app.together_api_client import TogetherMixtralClient
from sqlalchemy.orm import Session
from sqlalchemy import (
    text,
    and_, or_, select
)

engine = get_engine()
user_id = 'HqAXhad3jrUWmPibnMf1xZczNIq2'


2024-06-19 09:48:41 - Connecting to database using TCP
2024-06-19 09:48:41 - Connected to database using TCP


[nltk_data] Downloading package punkt to /home/eboraks/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


2024-06-19 09:48:45 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2024-06-19 09:48:45 - Starting new HTTPS connection (1): huggingface.co:443
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json HTTP/1.1" 200 0
2024-06-19 09:48:45 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/co

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
docs = getter.get_documents_by_user_id(user_id)

In [3]:
d1 = docs[2]
sentences = util.original_text_to_sentences(d1.original_text)
new_text = util.sentences_to_text(sentences)

In [4]:
d1.original_text

'When you select text while you’re reading, it’ll appear here.\nThe Trade Desk\'s Premium Internet Explained\nDuring The Trade Desk\'s Q1 earnings call, CEO Jeff Green slipped in the phrase "premium Internet" five times. Nobody from The Trade Desk mentioned the premium Internet once during its previous Q4 2023 earnings call.\nA few weeks later, the company released its Seller\'s and Publisher\'s Report with an accompanying subtitle of "The Rise of The Premium Internet." From the report:\nThe Trade Desk is pivoting from positioning itself as the DSP of the open Internet to the go-to DSP for the premium Internet. But why are they doing this?\nPremium Internet vs Walled Gardens\nThe Trade Desk wants advertisers to stop thinking about an "open Internet" full of rinky-dink display banners, made-for-advertising garbage, and non-transparent inventory.\nInstead, The Trade Desk desires to reframe inventory available via its platforms as a viable alternative to the walled gardens comprised of hi

In [5]:
new_text

'[0] When you select text while you’re reading, it’ll appear here. [1] The Trade Desk\'s Premium Internet Explained [2] During The Trade Desk\'s Q1 earnings call, CEO Jeff Green slipped in the phrase "premium Internet" five times. [3] Nobody from The Trade Desk mentioned the premium Internet once during its previous Q4 2023 earnings call. [4] A few weeks later, the company released its Seller\'s and Publisher\'s Report with an accompanying subtitle of "The Rise of The Premium Internet." [5] From the report: [6] The Trade Desk is pivoting from positioning itself as the DSP of the open Internet to the go-to DSP for the premium Internet. [7] But why are they doing this? [8] Premium Internet vs Walled Gardens [9] The Trade Desk wants advertisers to stop thinking about an "open Internet" full of rinky-dink display banners, made-for-advertising garbage, and non-transparent inventory. [10] Instead, The Trade Desk desires to reframe inventory available via its platforms as a viable alternative

In [6]:
client = TogetherMixtralClient()

In [7]:
res = await client.generate(DocumentPromptVerbatim, DocumentPromptVerbatim.get_messages(new_text))

2024-06-19 09:48:48 - Attempt 1 to generate summary
2024-06-19 09:49:27 - Response status: finished
2024-06-19 09:49:27 - Answer is not None. Stop retrying. Number of attempts 1


In [8]:
print(res.whatThisArticleIsAbout.answer)
print(res.whatThisArticleIsAbout.sentences_indcies) 
for ind in res.whatThisArticleIsAbout.sentences_indcies:
    print(sentences[ind])

This article discusses The Trade Desk's shift towards a 'premium Internet' as an alternative to walled gardens, aiming to provide high-quality and authenticated CTV video and digital audio placements for advertisers
[0, 1, 6, 15]
When you select text while you’re reading, it’ll appear here.
The Trade Desk's Premium Internet Explained
The Trade Desk is pivoting from positioning itself as the DSP of the open Internet to the go-to DSP for the premium Internet.
The company wants advertisers to believe its available inventory is preferable to that of the walled gardens to, as Mr. Green says, "gain more than our fair share of the nearly $1 trillion advertising TAM."
